# What is LSTM?

LSTM (Long Short-Term Memory) is a special type of Recurrent Neural Network (RNN) designed to:

✅ remember important information for a long time
✅ forget irrelevant information
✅ solve the vanishing gradient problem of simple RNNs

Used in:

Sentiment analysis

Machine translation

Speech recognition

Time-series forecasting

## Problem with Simple RNN (why LSTM exists)

Simple RNNs:

Forget old information quickly

Gradients become very small during backpropagation

Cannot learn long-term dependencies

Example:

“I watched the movie yesterday. It was amazing.”

## LSTM Solution (big idea)

LSTM introduces a memory cell and gates to control information flow.

🧠 Think of LSTM as:

A smart memory unit with switches

## LSTM Architecture (Main Components)

Each LSTM cell has:

1️⃣ Cell State (Ct) → long-term memory
2️⃣ Hidden State (ht) → short-term output
3️⃣ Three Gates:

Forget Gate

Input Gate

Output Gate

## LSTM Step-by-Step Working

At time step t, LSTM receives:

Input: xt

Previous hidden state: h(t-1)

Previous cell state: C(t-1)

## Step 1: Forget Gate 🚪

👉 Decides what to forget from old memory

Formula:
ft = σ(Wf · [ht-1, xt] + bf)

Meaning:

σ (sigmoid) → output between 0 and 1

0 → forget completely

1 → keep completely

📌 Example:
Forget old topic when sentence changes

## Step 2: Input Gate ✍️

👉 Decides what new information to store

Two parts:
(a) Input gate layer
it = σ(Wi · [ht-1, xt] + bi)

(b) Candidate memory
C̃t = tanh(Wc · [ht-1, xt] + bc)


it → how much to write

C̃t → new candidate values

## Step 3: Update Cell State 🧠

👉 Update long-term memory

Formula:
Ct = ft * C(t-1) + it * C̃t


Old memory × forget gate

New memory × input gate

📌 This is the core strength of LSTM

## Step 4: Output Gate 📤

👉 Decides what to output

Formula:
ot = σ(Wo · [ht-1, xt] + bo)

Hidden state:
ht = ot * tanh(Ct)


ht → passed to next time step

Also used as model output

In [1]:
import numpy as np
from tensorflow.keras.models import Sequential  # Sequential is the simplest neural network model in Keras, Layers are stacked one after another in a straight line
from tensorflow.keras.layers import Embedding, LSTM, Dense  # Converts word IDs → dense vectors

In [2]:
# 100 samples
# Each sample has 10 time-steps (words)
X = np.random.randint(1000, size=(100, 10))  # 100 samples (sentences),10 tokens per sample,Each number is a word index (as used by an Embedding layer)

# Binary output
y = np.random.randint(2, size=(100, 1)) # Generates random 0 or 1,One label per input sample

In [3]:
model = Sequential()

# Convert words to vectors
model.add(Embedding(input_dim=1000, output_dim=64, input_length=10))  # input_dim=1000 → vocabulary size (tokens from 0 to 999),output_dim=64 → each word becomes a 64-dim vector,input_length=10 → each sentence has 10 tokens

# LSTM layer (this is the RNN part)
model.add(LSTM(64))  # Reads the sequence one token at a time,Captures order and context

# Output layer
model.add(Dense(1, activation='sigmoid'))  # Converts LSTM output → prediction,sigmoid squashes value into [0, 1]

C:\Users\DELL\anaconda3\Lib\site-packages\keras\src\layers\core\embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [4]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [5]:
model.fit(X, y, epochs=5, batch_size=16)

Epoch 1/5
7/7 ━━━━━━━━━━━━━━━━━━━━ 6s 15ms/step - accuracy: 0.3735 - loss: 0.6940
Epoch 2/5
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.8429 - loss: 0.6855
Epoch 3/5
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.9838 - loss: 0.6716
Epoch 4/5
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.9837 - loss: 0.6392
Epoch 5/5
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.9782 - loss: 0.5489 


In [6]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding (Embedding)                │ (None, 10, 64)              │          64,000 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm (LSTM)                          │ (None, 64)                  │          33,024 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 1)                   │              65 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 291,269 (1.11 MB)

 Trainable params: 97,089 (379.25 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 194,180 (758.52 KB)

In [7]:
model = Sequential()
model.add(Embedding(1000, 64, input_length=10))
model.add(LSTM(64, return_sequences=True))
model.add(LSTM(32))
model.add(Dense(1, activation='sigmoid'))

# SENTIMENT ANALYSIS USING LSTM

In [2]:
# Import Required Libraries
import numpy as np
import pandas as pd
import re  # re = Regular Expressions, Remove punctuation,Remove numbers,Clean unwanted symbols,Pattern matching
import nltk

from nltk.corpus import stopwords  # Stopwords are common words that do not add meaning to text
from nltk.stem import PorterStemmer  # Stemming reduces words to their root/base form

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.preprocessing.text import Tokenizer  # Tokenizer converts text into integer sequences  
from tensorflow.keras.preprocessing.sequence import pad_sequences

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [3]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [4]:
data = {
    "review": [
        "I loved this movie, it was fantastic",
        "Worst movie ever, very boring",
        "Amazing acting and great story",
        "I hate this film",
        "Excellent movie, highly recommend",
        "Terrible plot and bad acting"
    ],
    "sentiment": [1, 0, 1, 0, 1, 0]
}

df = pd.DataFrame(data)
df

,review,sentiment
0,"I loved this movie, it was fantastic",1
1,"Worst movie ever, very boring",0
2,Amazing acting and great story,1
3,I hate this film,0
4,"Excellent movie, highly recommend",1
5,Terrible plot and bad acting,0


In [5]:
# Text Cleaning (MOST IMPORTANT IN NLP)
#Lowercase,Remove punctuation,Remove stopwords,Stemming

ps = PorterStemmer()  # Creates a stemming object,Used to reduce words to their root/base form
stop_words = set(stopwords.words('english'))  # Loads all English stopwords from NLTK,Converts them into a set for faster lookup (O(1))

def clean_text(text):  # Defines a reusable text-cleaning function
    text = text.lower()  # Converts everything to lowercase,Prevents treating Love and love as different words
    text = re.sub(r'[^a-z]', ' ', text)  # pattern → what to look for,replacement → what to replace it with 
    words = text.split()  # Splits the string into a list of words,Uses whitespace (spaces, tabs, newlines) as the separator by default
    words = [ps.stem(w) for w in words if w not in stop_words]  # each word w in words,ignore it if it’s a stopword,otherwise stem it,and collect the results in a new list
    return " ".join(words)  # Takes a list of words,Joins them into one single string,Puts a space between each word

df['clean_review'] = df['review'].apply(clean_text)  # Contains raw text reviews, Applies clean_text() row by row,Each review is passed as text to your function
df

,review,sentiment,clean_review
0,"I loved this movie, it was fantastic",1,love movi fantast
1,"Worst movie ever, very boring",0,worst movi ever bore
2,Amazing acting and great story,1,amaz act great stori
3,I hate this film,0,hate film
4,"Excellent movie, highly recommend",1,excel movi highli recommend
5,Terrible plot and bad acting,0,terribl plot bad act


In [6]:
# Tokenization (Text → Numbers)
tokenizer = Tokenizer(num_words=5000)  # Tokenizer(num_words=5000)Build a vocabulary of the top 5000 most frequent words
tokenizer.fit_on_texts(df['clean_review']) # Tokenizer scans all cleaned reviews and builds

sequences = tokenizer.texts_to_sequences(df['clean_review'])  # Converts text → integer sequences

In [7]:
# Padding (Make Same Length)
max_len = 20  # Maximum number of tokens per review, # Short sequences → padded,Long sequences → truncated

X = pad_sequences(
    sequences,
    maxlen=max_len,
    padding='post'
)
  
y = np.array(df['sentiment'])  # Takes the sentiment column from your DataFrame,Converts it into a NumPy array,Stores it in y your target variabl

In [8]:
# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

In [9]:
# Build LSTM Model (REAL ARCHITECTURE)
model = Sequential()

model.add(Embedding(  # Takes word IDs (integers from your Tokenizer),Converts each ID into a dense vector,Learns word meaning during training
    input_dim=5000,  # input_dim = 5000,Size of the vocabulary,Must match Tokenizer(num_words=5000)
    output_dim=128,  # Dimension of each word vector,Each word becomes a 128-number vector
    input_length=max_len
))

model.add(LSTM(128, return_sequences=False))
model.add(Dropout(0.5))

model.add(Dense(1, activation='sigmoid'))

C:\Users\DELL\anaconda3\Lib\site-packages\keras\src\layers\core\embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [10]:
# Compile Model
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [11]:
# Train Model
history = model.fit(
    X_train,
    y_train,
    epochs=10,
    batch_size=2,
    validation_data=(X_test, y_test)
)

Epoch 1/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 9s 1s/step - accuracy: 0.5000 - loss: 0.7055 - val_accuracy: 0.5000 - val_loss: 0.6932
Epoch 2/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 216ms/step - accuracy: 0.5000 - loss: 0.7034 - val_accuracy: 0.5000 - val_loss: 0.6932
Epoch 3/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 213ms/step - accuracy: 0.3333 - loss: 0.6949 - val_accuracy: 0.5000 - val_loss: 0.6938
Epoch 4/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 208ms/step - accuracy: 0.5000 - loss: 0.6659 - val_accuracy: 0.5000 - val_loss: 0.6937
Epoch 5/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 273ms/step - accuracy: 0.3333 - loss: 0.7176 - val_accuracy: 0.5000 - val_loss: 0.6935
Epoch 6/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 355ms/step - accuracy: 0.5000 - loss: 0.7128 - val_accuracy: 0.5000 - val_loss: 0.6934
Epoch 7/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 389ms/step - accuracy: 0.1667 - loss: 0.7011 - val_accuracy: 0.5000 - val_loss: 0.6933
Epoch 8/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 362ms/step - accuracy: 0.1667 - loss: 0.7221 - val_accuracy: 0.5000 - val_loss: 0.

In [12]:
# Evaluate Model
y_pred = (model.predict(X_test) > 0.5).astype(int)

print("Accuracy:", accuracy_score(y_test, y_pred))

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 981ms/step
Accuracy: 0.5


In [19]:
# Prediction Function (Production Style)
def predict_sentiment(text):
    text = clean_text(text)  # Applies the same preprocessing used during training
    seq = tokenizer.texts_to_sequences([text])  # Converts cleaned text → word IDs,Wrapped in a list because Keras expects batch input
    padded = pad_sequences(seq, maxlen=max_len, padding='post')
    pred = model.predict(padded)[0][0]  # [0][0] extracts the scalar probability
    
    return "Positive" if pred > 0.5 else "Negative"
print(predict_sentiment("This movie was awesome"))
print(predict_sentiment("Very bad and boring movie"))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 128ms/step
Positive
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 140ms/step
Positive
